# 20 — Final End-to-End Pipeline Integration

**SIH PS 26168 — Intelligent Dead Reckoning**

> **Roadmap Sections 33 & 34:**
> - Unified inference/orchestration pipeline combining all 10 navigation components
> - Validates seamless execution on held-out session `S1` (Driver A)
> - Verifies clean state transitions: `FULL_FUSION` $\leftrightarrow$ `DEGRADED_GNSS` $\leftrightarrow$ `GNSS_BLACKOUT` $\to$ `RECOVERY`

## 1. Environment & Pipeline Initialization

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.integration.final_navigation_pipeline import FinalNavigationPipeline
from src.preprocessing.data_loader import IOVNBDLoader
from src.calibration.alignment import PhoneVehicleAlignment

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Integration Test Running on Device: {device}')

fixed_knet = PROJECT_ROOT / 'checkpoints' / 'kalmannet_fixed_input' / 'kalmannet_best.pt'
base_knet  = PROJECT_ROOT / 'checkpoints' / 'kalmannet' / 'kalmannet_best.pt'
knet_ckpt  = str(fixed_knet if fixed_knet.exists() else base_knet)
map_gnn_ckpt = str(PROJECT_ROOT / 'checkpoints' / 'map_gnn' / 'map_gnn_best.pt')
graph_path = str(PROJECT_ROOT / 'data' / 'OSM' / 'road_graph_coventry.pkl')

pipeline = FinalNavigationPipeline(
    knet_checkpoint_path=knet_ckpt,
    map_gnn_checkpoint_path=map_gnn_ckpt,
    road_graph_path=graph_path,
    device=device
)
print('Unified FinalNavigationPipeline successfully constructed.')

## 2. Load Driving Session S1 & Calibrate Initial Alignment

In [ ]:
loader = IOVNBDLoader()
sess = loader.load_session('S1', preprocess_imu=True)

acc_raw = sess['accel_filtered']
gyr_raw = sess['gyro_filtered']
lat_gps = sess['gps']['lat']
lon_gps = sess['gps']['lon']
enu_gt = sess['enu_coords'][:, :2]
veh_spd = sess['vehicle']['speed_mps']
if veh_spd is None:
    veh_spd = sess['gps']['speed_mps']
if veh_spd is None:
    veh_spd = np.zeros(len(enu_gt))

# Calibrate phone-to-vehicle rotation matrix R_p2v
aligner = PhoneVehicleAlignment()
R_p2v = aligner.calibrate(sess['accel_raw'], zupt_mask=sess['zupt_mask'], velocity_ref=veh_spd)
print('Calibrated Phone-to-Vehicle Rotation Matrix:')
print(np.round(R_p2v, 4))

# Initialize pipeline
pipeline.initialize(
    lat0=sess['gps']['lat0'],
    lon0=sess['gps']['lon0'],
    alt0=sess['gps']['alt0'],
    initial_heading=0.0,
    initial_speed=float(veh_spd[0]),
    R_p2v=R_p2v
)
print('Pipeline initialized at Geodetic Origin.')

## 3. Execute 1,000-Step End-to-End Simulation with Tunnel Blackout

In [ ]:
SIM_LEN = 1000
trajectory_history = []
modes_history = []

# Simulate 30s blackout between step 400 and 700
BLACKOUT_START = 400
BLACKOUT_END = 700

print(f'Running end-to-end integration simulation for {SIM_LEN} steps (100.0 seconds)...')
for k in range(SIM_LEN):
    is_bo = (BLACKOUT_START <= k < BLACKOUT_END)
    p_gnss = (lat_gps[k], lon_gps[k]) if not is_bo else None

    state = pipeline.step(
        accel_raw=acc_raw[k],
        gyro_raw=gyr_raw[k],
        p_gnss_geodetic=p_gnss,
        hdop=1.2,
        is_blackout=is_bo,
        speed_ref=float(veh_spd[k]),
        dt=0.1
    )
    trajectory_history.append((state['east_m'], state['north_m']))
    modes_history.append(state['mode'])

est_pos = np.array(trajectory_history)
gt_eval = enu_gt[:SIM_LEN] - enu_gt[0]
err = np.linalg.norm(est_pos - gt_eval, axis=1)
rmse = float(np.sqrt(np.mean(err**2)))

print('=' * 60)
print('  PIPELINE INTEGRATION TEST COMPLETED')
print('=' * 60)
print(f'Total Steps Simulated      : {SIM_LEN} (10 Hz)')
print(f'Overall Position RMSE      : {rmse:.2f} meters')
print(f'Final Position Error       : {err[-1]:.2f} meters')
print(f'Blackout Steps Handled     : {BLACKOUT_END - BLACKOUT_START} steps (30 seconds)')
print(f'Distinct Modes Observed    : {set(modes_history)}')
print('=' * 60)